> The training data were computed with Thermo-Calc, and its license does not allow us to redistribute them, so
> **no dataset, tokenizer or trained weights are provided**, and this notebook cannot be re-run as is. Users with a
> Thermo-Calc license can generate data in the same format (see "What you need to provide" in the README) and
> train the model with this notebook.


## 1. Check the environment

In [1]:

import os, sys
print('Here is : \n', os.popen('pwd').read())
print('What we get in hardware: \n', os.popen('nvidia-smi').read())
kernel_name = os.path.basename(sys.executable.replace("/bin/python",""))
print ("The VirEnv kernal in action: ", kernel_name)
#
# # make it work for both py and notebook
# try:
#     from jupyter_client import kernelspec
#     spec = kernelspec.get_kernel_spec(kernel_name)
#     print("Path to it: ", spec.resource_dir)
# except:
#     print ("This suppose to be a .py run")
# # /path/to/my/kernel
import torch
print("What we have in software: \n Torch version:", torch.__version__)
print('Python: ', sys.version) # no switch case code
#
print('What hardware the software see:')
device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)
print(device)
num_of_gpus = torch.cuda.device_count()
print("# of GPU", num_of_gpus)
#
torch.cuda.empty_cache()

Here is : 
 /trace/group/tmousavi/jixuand/AlloyGPT

What we get in hardware: 
 Fri Dec 19 11:55:47 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 545.23.08              Driver Version: 545.23.08    CUDA Version: 12.3     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A40                     On  | 00000000:25:00.0 Off |                    0 |
|  0%   37C    P0              70W / 300W |   2004MiB / 46068MiB |      0%      Default |
|                                         |                      |             

## 2. Configurate the task

In [2]:
# for debug
import importlib

In [3]:
# load in source code packages

import AlloyGPT.UtilityPack as UtilityPack
importlib.reload(UtilityPack)
#
import AlloyGPT.DataPack_v5 as DataPack
importlib.reload(DataPack)
#
import AlloyGPT.ModelPack_v3 as ModelPack
importlib.reload(ModelPack)
#
import AlloyGPT.TrainPack_v3 as TrainPack
importlib.reload(TrainPack)
#
import AlloyGPT.TestPack_v3 as TestPack
importlib.reload(TestPack)

<module 'AlloyGPT.TestPack_v3' from '/trace/group/tmousavi/jixuand/AlloyGPT/AlloyGPT/TestPack_v3.py'>

In [4]:
# load in additional packages

import pickle
from matplotlib import pyplot as plt
import seaborn as sns
from tqdm import tqdm
import math
import pandas as pd
# ++
import numpy as np
import getpass

In [5]:
# 
# code_dir = '/content/AlloyGPT_InternalTest_0/' # for colab run
# 
code_dir = './' # for local run

In [6]:
# control key

# ===============================================
# Global control key setup: may into yaml file
# ===============================================
# control key to be shared everywhere
#
CKeys = {}
CKeys['Working_Mode'] = 1 # 1 # 0 # 1 # 2
# 0: prepare dataset: run only on single node
# 1: training,
# 2: testing

# only for restart of training part
CKeys['IF_FirstRun'] = 1 # 1 # 2
CKeys['Resume_from_where'] = 'LAST'
# 1: train for 1st run; 2: other training loop

CKeys['Problem_ID'] = 6
# 1: Transformer-based causal LM: GPT2
# 2: Mamba-based causal LM
# 3: T-based causal LM + HF dataset on AlloyLan
# 4: selfmade S6
# 5: BPE tokenizer + hf-S6
# 6: BPE tokenizer + GPT2

# where to pick up the checkpoint
CKeys['Resume_from_where'] = "LAST" # "BEST"

# On Debug
# CKeys['Debug']=1
CKeys['Debug']=0

if CKeys['Debug'] == 1:
    CKeys['Debug_Data'] = 1
    CKeys['Debug_Model'] = 1
    CKeys['Debug_Train'] = 1
    CKeys['Debug_Test'] = 1
    # for On cluster run
    CKeys['if_slient_run']=0
else:
    CKeys['Debug_Data']=0
    CKeys['Debug_Model']=0
    CKeys['Debug_Train']=0
    CKeys['Debug_Test']=0
    # for On cluster run
    CKeys['if_slient_run']=1

In [7]:
# if CKeys['IF_FirstRun']==1:
    
#     dataset_key = getpass.getpass(prompt="Please enter your key to the dataset: \n\n")

In [8]:
print ("Check CKeys: \n")
for this_key in CKeys.keys():
    print (f"{this_key}: {CKeys[this_key]}")

Check CKeys: 

Working_Mode: 1
IF_FirstRun: 1
Resume_from_where: LAST
Problem_ID: 6
Debug: 0
Debug_Data: 0
Debug_Model: 0
Debug_Train: 0
Debug_Test: 0
if_slient_run: 1


In [9]:
# problem review:
print ("Problem Review: ")
print('Problem type: ', CKeys['Problem_ID'])
print('Debug mode: ', CKeys['Debug'])
print('Working mode: ', CKeys['Working_Mode'])

Problem Review: 
Problem type:  6
Debug mode:  0
Working mode:  1


In [10]:
# ===============================================
# Parameter Keys: may think of moving this into
# a yaml file
# 1. set for the first time
# 2. reload in sequential runs
# ===============================================
#
PKeys = {}
PKeys['prefix']='Causal_LM_TwoWays'

PKeys['wk_path']='./training_resu/'
#
# may add keys for lower level
# ||||||||||||||||||||||||||||||||||||||||||||||||||||||
# on data + tokenizer
Data_PKeys = {}
Data_PKeys['data_dir']=PKeys['wk_path']+'/0_dataprocess/'
Data_PKeys['tokenizer_dir']=Data_PKeys['data_dir']+'/tokenizer'
# problem-specified ones
Data_PKeys['tokenizer_type']='BPE_customerized_0' # 'Byte_Tokenizer'
# Data_PKeys['tokenizer_file']='/trace/group/tmousavi/bni2/1_JupyterGit/workspace/Test_NanoGPT_1/Local_Store/cBPE_tokenizer_ver1.json'
Data_PKeys['tokenizer_file']=code_dir+'/assets/cBPE_tokenizer_ver1.json'
Data_PKeys['train_ratio']=0.900
Data_PKeys['valid_ratio']=0.001
Data_PKeys['fix_random']=12345
Data_PKeys['context_length']=208 # length of text window
# not working for google colab T4
Data_PKeys['batch_size'] = 12 # 12 # 8 # 4 # 8 # 16 # 24 # 32
#
# Data_PKeys['batch_size'] = 4 # 12 # 8 # 4 # 8 # 16 # 24 # 32
# # use hf-dataset:
# Data_PKeys['hf_data_repo'] = 'Bo-Ni/Al_alloy_CALPHAD_test_4_full' # 'Bo-Ni/Al_alloy_CALPHAD_test_3_full'
# use local dataset:
# Data_PKeys['local_data_file'] = 'Training_data/IN718_with_variability_1pc.arrow'
Data_PKeys['local_data_file'] = 'Training_data/Updated_local_variation/df_results_010_var.arrow'
#
# ||||||||||||||||||||||||||||||||||||||||||||||||||||||
# on model
Model_PKeys = {}
Model_PKeys['model_dir']=PKeys['wk_path']+'/1_model/'
Model_PKeys['model_type'] = 6 # 5: hfS6; 6: GPT2
Model_PKeys['model_args'] = dict(
    vocab_size=256, # len of the dict of the tokenizer
    n_layer = 36, # 24 # 6 # num of MHA blocks
    block_size = 1024,
    n_embd = 1024,
    # inside one MHA layer
    n_head = 16,
    bias = False,
    dropout = 0.2,
)
#
# ||||||||||||||||||||||||||||||||||||||||||||||||||||||
# on training
Train_PKeys = {}
Train_PKeys['out_dir'] = Model_PKeys['model_dir'] + "/training_dir/"
# + customize ones +
Train_PKeys['batch_size'] = Data_PKeys['batch_size']
Train_PKeys['block_size'] = Data_PKeys['context_length']
Train_PKeys['vocab_size'] = Model_PKeys['model_args']['vocab_size']
Train_PKeys['num_train_epochs'] = 1 if CKeys['Debug']==1 else 5 # 2
Train_PKeys['gradient_accumulation_steps'] = 8 # 40 # 1 # this's GAS
#
# AdamW optimizer, may put this into model class
Train_PKeys['weight_decay'] = 1e-1
Train_PKeys['beta1'] = 0.9
Train_PKeys['beta2'] = 0.95
# 0.99  # make a bit bigger because number of tokens per iter is small
Train_PKeys['grad_clip'] = 1.0 # clip gradients at this value, or disable if == 0.0
#
# Learning Plan:
# (linear warmup) + (cosine schedule) for lr_decay_iters + constant at min_lr
# Train_PKeys['max_iters'] = math.ceil(
#     len(train_dataloader)/Train_PKeys['gradient_accumulation_steps']
# ) * Train_PKeys['num_train_epochs'] # in GAS
# Train_PKeys['lr_decay_iters'] = Train_PKeys['max_iters']
Train_PKeys['warmup_iters'] = 1_000 # for large one # in GAS
# Train_PKeys['learning_rate'] = 6e-4 #original learning rate
Train_PKeys['learning_rate'] = 1e-4
Train_PKeys['decay_lr'] = True
# whether to decay the learning rate
# True: linear warmup + cosine decay + constant
# False: constant
#
Train_PKeys['min_lr'] = Train_PKeys['learning_rate']*1.e-1
#
# reporting and recording
Train_PKeys['report_1_trai_loss_this_GAS'] = 2 if CKeys['Debug']==1 else 20
Train_PKeys['report_2_vali_pred_this_GAS'] = 2 if CKeys['Debug']==1 else 200
Train_PKeys['report_3_save_mode_this_GAS'] = \
    Train_PKeys['report_2_vali_pred_this_GAS']*2 \
    if CKeys['Debug']==1 else \
    Train_PKeys['report_2_vali_pred_this_GAS']*2
#
# OTHERS: parallel setup
# DDP settings
Train_PKeys['backend'] = 'nccl' # 'nccl', 'gloo', etc.
# system
Train_PKeys['device'] = 'cuda' # examples: 'cpu', 'cuda', 'cuda:0', 'cuda:1' etc., or try 'mps' on macbooks
Train_PKeys['dtype'] = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16' # 'float32', 'bfloat16', or 'float16', the latter will auto implement a GradScaler
Train_PKeys['compile'] = True # use PyTorch 2.0 to compile the model to be faster
#

# ||||||||||||||||||||||||||||||||||||||||||||||||||||||
# on testing
Test_PKeys = {}
Test_PKeys['out_dir'] = Model_PKeys['model_dir'] + "/test_dir/"
#
# TBA: useful keys for testing
# do a initialization
Test_PKeys['num_samples'] = 2 # 10
# number of samples to draw
Test_PKeys['max_new_tokens'] = 1024 # 500
# number of tokens generated in each sample
Test_PKeys['temperature'] = 2
# 1.0 = no change, < 1.0 = less random, > 1.0 = more random, in predictions
Test_PKeys['top_k'] = 200
# retain only the top_k most likely tokens, clamp others to have 0 probability
Test_PKeys['seed'] = 1337
Test_PKeys['device'] = Train_PKeys['device']
# examples: 'cpu', 'cuda', 'cuda:0', 'cuda:1', etc.
Test_PKeys['dtype'] = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'
# 'float32' or 'bfloat16' or 'float16'
Test_PKeys['compile'] = Train_PKeys['compile']

# pack keys for later
PKeys['Data_PKeys'] = Data_PKeys
PKeys['Model_PKeys'] = Model_PKeys
PKeys['Train_PKeys'] = Train_PKeys
PKeys['Test_PKeys'] = Test_PKeys


In [11]:
# ===============================================
# setup folder structures and parameter files
# place_holder
# ===============================================
print ("===============================================")
print ("setup folder structures and parameter files")
print ("===============================================")
#
# 1. global dir
print ('Working path exists or not: ', os.path.exists(PKeys['wk_path']))
if not os.path.exists(PKeys['wk_path']):
    UtilityPack.create_path(PKeys['wk_path'])

# 2. some global keys for recalling
# store info for later
PKeys['pk_data_pack']=PKeys['wk_path']+'/data_pack.pickle'
PKeys['pk_model_pack']=PKeys['wk_path']+'/model_pack.pickle'
PKeys['pk_train_pack']=PKeys['wk_path']+'/train_pack.pickle'


setup folder structures and parameter files
Working path exists or not:  True


In [12]:
print (f"Working path: ", PKeys['wk_path'])

Working path:  ./training_resu/


In [13]:
#
# clean EVERYTHING in the dir if 1st
#
# if CKeys['Working_Mode']==1 and CKeys['IF_FirstRun']==1:
print ("===============================================")
print ("Clean EVERYTHING in the dir if 1st...")
print ("===============================================")
#
if CKeys['Working_Mode']==1 and CKeys['IF_FirstRun']==1:
    #
    if os.path.exists(PKeys['wk_path']):
        cmd_line=f"rm -r {PKeys['wk_path']}"
        print("clean the slade...")
        print(f"excute {cmd_line}")
        os.popen(cmd_line).read()
        #
    # create dir for working space
    UtilityPack.create_path(PKeys['wk_path'])

Clean EVERYTHING in the dir if 1st...
Creating the given path...
Done.


## 3. On dataset

In [14]:
# load in
# use HF infurstructures
from datasets import (
    load_dataset,
    load_from_disk,
    load_dataset_builder,
    get_dataset_split_names,
    DatasetDict,
    load_from_disk,
)
from torch.utils.data.dataloader import DataLoader

In [15]:
import datasets
print(datasets.__version__)

3.3.2


In [16]:
# ===================================================
# setup the dataset key
# ===================================================
#
if CKeys['Working_Mode']==1 and CKeys['IF_FirstRun']==1:
    print ("===================================================")
    print ("setup the dataset key")
    print ("===================================================")

    # first initialize the key
    DataKeys={}
    # pass on the keys
    print ("Get the following keys...")
    for this_key in PKeys['Data_PKeys'].keys():
        print ("{}: \n{}".format(this_key, PKeys['Data_PKeys'][this_key]))
        DataKeys[this_key] = PKeys['Data_PKeys'][this_key]

    print ("Create folders if needed...")
    # 1. creat subdir for datapart
    UtilityPack.create_path(DataKeys['data_dir'])
    # tokenizer folder
    UtilityPack.create_path(DataKeys['tokenizer_dir'])

setup the dataset key
Get the following keys...
data_dir: 
./training_resu//0_dataprocess/
tokenizer_dir: 
./training_resu//0_dataprocess//tokenizer
tokenizer_type: 
BPE_customerized_0
tokenizer_file: 
.//assets/cBPE_tokenizer_ver1.json
train_ratio: 
0.9
valid_ratio: 
0.001
fix_random: 
12345
context_length: 
208
batch_size: 
12
local_data_file: 
Training_data/Updated_local_variation/df_results_010_var.arrow
Create folders if needed...
Creating the given path...
Done.
Creating the given path...
Done.


In [ ]:
# ===================================================
# process the dataset
# ===================================================
#
if CKeys['Working_Mode']==1 and CKeys['IF_FirstRun']==1:
    print ("===================================================")
    print ("process the dataset")
    print ("===================================================")
    # process
    # 1. download the dict from HF
    dataset_dict = load_dataset(
        "arrow", 
        data_files={'train': DataKeys['local_data_file']}
    )
    
    print ("raw dataset:\n")
    print (dataset_dict)

    
    # 2.seperate into three
    dataset_dict_sepe = DataPack.data_dict_seperate_train_vali_test(
        dataset_dict,
        train_ratio=DataKeys['train_ratio'],
        vali_ratio=DataKeys['valid_ratio'],
        if_seed=(DataKeys['fix_random']>0),
        seed=DataKeys['fix_random'] if DataKeys['fix_random']>0 else None
    )
    print ("seperate raw dataset:\n")
    print (dataset_dict_sepe)
    # 3. do statistics on the dataset


    # 4. create the tokenizer
    tokenizer = DataPack.build_tokenizer(
        tokenizer_type=DataKeys['tokenizer_type'],
        tokenizer_file=DataKeys['tokenizer_file'],
        seq_len=DataKeys['context_length'],
    )

    # 5. convert raw data into text data for GPT
    # a. number record ==> sentences
    sentence_dataset_dict_sepe = dataset_dict_sepe.map(
        # self-defined function
        DataPack.assemble_multi_sentence,
        # other keys besides element in the procssing fun
        fn_kwargs={
            "key_list": ['Pred001', 'Gene001'],
        },
        batched= False, # True, # Note, sentences don't work for batched
        # only keep those for training by removing the old ones
        remove_columns=dataset_dict_sepe["train"].column_names,
    )
    # b. sentences ==> tokenized sentences
    tokenized_dataset_dict_sepe = sentence_dataset_dict_sepe.map(
        # self-defined function
        DataPack.tokenize_multi_sentence_with_BPE,
        # other keys
        fn_kwargs={
            "key_list": ['Pred001', 'Gene001'],
            "tokenizer": tokenizer,
            "context_length": DataKeys['context_length'],
        },
        #
        batched = True,
        # only keep those for training by removing the old ones
        remove_columns=sentence_dataset_dict_sepe["train"].column_names,
    )
    
    # c. convert into dataloader
    tokenized_dataset_dict_sepe.set_format(type="torch")
    # build into pytorch object
    #
    train_dataloader = DataLoader(
        tokenized_dataset_dict_sepe["train"],
        batch_size=DataKeys['batch_size'], # 32,
        shuffle=False,
    )
    #
    eval_dataloader = DataLoader(
        tokenized_dataset_dict_sepe["valid"],
        batch_size=DataKeys['batch_size'], # 32,
        shuffle=True,
    )
    # use for test
    #
    test_dataloader = DataLoader(
        tokenized_dataset_dict_sepe["test"],
        batch_size=DataKeys['batch_size'], # 32,
        shuffle=False,
    )

    # store the key and results
else:
    # load back DataKeys
    pass


process the dataset


Generating train split: 0 examples [00:00, ? examples/s]

raw dataset:

DatasetDict({
    train: Dataset({
        features: ['NiMassPC', 'CrMassPC', 'NbMassPC', 'MoMassPC', 'TiMassPC', 'AlMassPC', 'CoMassPC', 'Mean YS', 'Std YS', 'Delta_YS_expected'],
        num_rows: 250000
    })
})
seperate raw dataset:

DatasetDict({
    train: Dataset({
        features: ['NiMassPC', 'CrMassPC', 'NbMassPC', 'MoMassPC', 'TiMassPC', 'AlMassPC', 'CoMassPC', 'Mean YS', 'Std YS', 'Delta_YS_expected'],
        num_rows: 225000
    })
    valid: Dataset({
        features: ['NiMassPC', 'CrMassPC', 'NbMassPC', 'MoMassPC', 'TiMassPC', 'AlMassPC', 'CoMassPC', 'Mean YS', 'Std YS', 'Delta_YS_expected'],
        num_rows: 250
    })
    test: Dataset({
        features: ['NiMassPC', 'CrMassPC', 'NbMassPC', 'MoMassPC', 'TiMassPC', 'AlMassPC', 'CoMassPC', 'Mean YS', 'Std YS', 'Delta_YS_expected'],
        num_rows: 24750
    })
})


Map:   0%|          | 0/225000 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Map:   0%|          | 0/24750 [00:00<?, ? examples/s]

Map:   0%|          | 0/225000 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Map:   0%|          | 0/24750 [00:00<?, ? examples/s]

In [18]:
import os
import numpy as np

# It's good practice to tell the tokenizers library to not use parallelism here
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Searching for the TRUE maximum token length by ignoring padding...")

def get_true_token_length(example):
    """
    Correctly calculates sentence length by summing the attention mask,
    which ignores padding tokens.
    """
    # Tokenize both sentences
    pred_encoding = tokenizer.encode(example['Pred001_sentence'])
    gene_encoding = tokenizer.encode(example['Gene001_sentence'])
    
    # The sum of the attention_mask gives the number of real (non-padding) tokens
    return {
        'pred_len': sum(pred_encoding.attention_mask),
        'gene_len': sum(gene_encoding.attention_mask)
    }

# Use .map() to apply the CORRECT function to the dataset
dataset_with_lengths = sentence_dataset_dict_sepe['train'].map(
    get_true_token_length,
    num_proc=4  # You can adjust this based on your CPU cores
)

# --- VERIFICATION STEP ---
# Print the first 5 rows to show the new length columns are correct
print("\n--- Verification: Displaying first 5 rows with their true lengths ---")
print(dataset_with_lengths.select(range(5)).to_pandas())
# ---

# Find the maximum length from the new columns
max_len = max(max(dataset_with_lengths['pred_len']), max(dataset_with_lengths['gene_len']))

print(f"\nThe longest sentence in your dataset TRULY has {max_len} tokens.")

# It's good practice to round up to a multiple of 8 or 16 for efficiency
new_block_size = (max_len + 15) & -16 
print(f"RECOMMENDED new block_size: {new_block_size}")

Searching for the TRUE maximum token length by ignoring padding...


Map (num_proc=4):   0%|          | 0/225000 [00:00<?, ? examples/s]


--- Verification: Displaying first 5 rows with their true lengths ---
                                    Pred001_sentence  \
0  {Task:Pred001}={Composition:[(Ti):+1.122e+00,(...   
1  {Task:Pred001}={Composition:[(Ti):+9.623e-01,(...   
2  {Task:Pred001}={Composition:[(Ti):+8.500e-01,(...   
3  {Task:Pred001}={Composition:[(Ti):+1.043e+00,(...   
4  {Task:Pred001}={Composition:[(Ti):+7.517e-01,(...   

                                    Gene001_sentence  pred_len  gene_len  
0  {Task:Gene001}={Property:[MeanYs:+1.342e+03,St...       189       189  
1  {Task:Gene001}={Property:[MeanYs:+1.114e+03,St...       189       189  
2  {Task:Gene001}={Property:[MeanYs:+1.178e+03,St...       189       189  
3  {Task:Gene001}={Property:[MeanYs:+1.288e+03,St...       189       189  
4  {Task:Gene001}={Property:[MeanYs:+9.612e+02,St...       189       189  

The longest sentence in your dataset TRULY has 189 tokens.
RECOMMENDED new block_size: 192


In [19]:
import random

print("\n--- ENHANCED SPOT-CHECK ---")

# Get the total number of training examples
num_examples = len(sentence_dataset_dict_sepe['train'])

# Select 3 random indices
random_indices = random.sample(range(num_examples), 3)

for i, index in enumerate(random_indices):
    example = sentence_dataset_dict_sepe['train'][index]
    print(f"\n--- Random Example #{i+1} (Index: {index}) ---")
    print("FORWARD:", example['Pred001_sentence'])
    print("INVERSE:", example['Gene001_sentence'])

print("\n---------------------------\n")


--- ENHANCED SPOT-CHECK ---

--- Random Example #1 (Index: 143049) ---
FORWARD: {Task:Pred001}={Composition:[(Ti):+8.858e-01,(Cr):+1.726e+01,(Mo):+3.078e+00,(Al):+5.862e-01,(Nb):+5.479e+00,(Co):+9.140e-02,(Ni):+5.266e+01]}=>{Property:[MeanYs:+1.223e+03,StdYS:+4.609e+01,DeltaYS:+1.322e+02]}</s>
INVERSE: {Task:Gene001}={Property:[MeanYs:+1.223e+03,StdYS:+4.609e+01,DeltaYS:+1.322e+02]}=>{Composition:[(Ti):+8.858e-01,(Cr):+1.726e+01,(Mo):+3.078e+00,(Al):+5.862e-01,(Nb):+5.479e+00,(Co):+9.140e-02,(Ni):+5.266e+01]}</s>

--- Random Example #2 (Index: 138541) ---
FORWARD: {Task:Pred001}={Composition:[(Ti):+7.532e-01,(Cr):+1.767e+01,(Mo):+3.046e+00,(Al):+3.509e-01,(Nb):+5.338e+00,(Co):+2.771e-01,(Ni):+5.032e+01]}=>{Property:[MeanYs:+1.002e+03,StdYS:+4.479e+01,DeltaYS:+1.238e+02]}</s>
INVERSE: {Task:Gene001}={Property:[MeanYs:+1.002e+03,StdYS:+4.479e+01,DeltaYS:+1.238e+02]}=>{Composition:[(Ti):+7.532e-01,(Cr):+1.767e+01,(Mo):+3.046e+00,(Al):+3.509e-01,(Nb):+5.338e+00,(Co):+2.771e-01,(Ni):+5.032

In [20]:
#
# save or recall necessary parts
#
if CKeys['Working_Mode']==1 and CKeys['IF_FirstRun']==1:
    print ("==========================================")
    print ("save the dataset and tokenizer....")
    print ("==========================================")

    # 1. on DataKeys
    with open(PKeys['pk_data_pack'], 'wb') as handle:
        pickle.dump(DataKeys, handle, protocol=pickle.HIGHEST_PROTOCOL)
    #
    # 2. on tokenizer
    # # --
    # tokenizer.save_pretrained(
    #     DataKeys['tokenizer_dir']
    # )
    # ++
    DataPack.save_tokenizer(
        DataKeys,
        tokenizer,
    )
    #
    # 3. dataloaders
    # for trainining
    torch.save(train_dataloader, DataKeys['data_dir']+'/train_dataloader.pt')
    torch.save(eval_dataloader, DataKeys['data_dir']+'/eval_dataloader.pt')
    torch.save(test_dataloader, DataKeys['data_dir']+'/test_dataloader.pt')

    # 4. something else: sentence_data_dict
    # hf_object
    sentence_dataset_dict_sepe.save_to_disk(
        DataKeys['data_dir']+'/sentence_dataset_dict_hf'
    )

else:
    print("==========================================")
    print ('This is not a data-prepare run')
    print ('Load back in the data packages...')
    print("==========================================")



    with open(PKeys['pk_data_pack'], 'rb') as handle:
        # data_pack = pickle.load(handle)
        DataKeys = pickle.load(handle)


    # on tokenizer
    # # --
    # from transformers import AutoTokenizer
    # tokenizer = AutoTokenizer.from_pretrained(
    #     DataKeys['tokenizer_dir']
    # )
    # ++
    tokenizer  = DataPack.reload_tokenizer(
        DataKeys
    )

    # on dataloaders
    train_dataloader = torch.load(DataKeys['data_dir']+'/train_dataloader.pt',weights_only=False)
    eval_dataloader = torch.load(DataKeys['data_dir']+'/eval_dataloader.pt',weights_only=False)
    test_dataloader = torch.load(DataKeys['data_dir']+'/test_dataloader.pt',weights_only=False)

    # something else:
    sentence_dataset_dict_sepe = load_from_disk(
        DataKeys['data_dir']+'/sentence_dataset_dict_hf'
    )

    print ('Done.')


save the dataset and tokenizer....


Saving the dataset (0/1 shards):   0%|          | 0/225000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/250 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/24750 [00:00<?, ? examples/s]

In [21]:
# train_dataloader

In [22]:
# os.path.exists(DataKeys['data_dir']+'/train_dataloader.pt')

## 4. On model

In [23]:
if CKeys['Working_Mode']==1 and CKeys['IF_FirstRun']==1:

    print ("======================================")
    print ("Initialize the model key")
    print ("======================================")
    ModelKeys = {}
    # pass on the keys
    print ("Get the following keys...")
    for this_key in PKeys['Model_PKeys'].keys():
        print ("{}: \n{}".format(this_key, PKeys['Model_PKeys'][this_key]))
        ModelKeys[this_key] = PKeys['Model_PKeys'][this_key]

    # 1. create the folder
    UtilityPack.create_path(ModelKeys['model_dir'])
    # 2. deliver one key for model building
    model_args = ModelKeys['model_args']


    print("==========================================")
    print("Save the model config ...")
    print("==========================================")
    # on ModelKeys
    with open(PKeys['pk_model_pack'], 'wb') as handle:
        pickle.dump(ModelKeys, handle, protocol=pickle.HIGHEST_PROTOCOL)

else:
    # this can be: 2nd training run or test run
    # load in the keys

    print("==========================================")
    print ('This is not the first run')
    print ('Load back in the model packages...')
    print("==========================================")

    with open(PKeys['pk_model_pack'], 'rb') as handle:
        # data_pack = pickle.load(handle)
        ModelKeys = pickle.load(handle)

    # unpack the model_args key
    model_args = ModelKeys['model_args']
    print ("model_args: \n", model_args)

    # From Trainer to know what parameters to load in
    with open(PKeys['pk_train_pack'], 'rb') as handle:
        # data_pack = pickle.load(handle)
        TrainKeys = pickle.load(handle)
    # # unpack the model_args key
    # model_args = ModelKeys['model_args']
    print ("TrainKeys: \n", TrainKeys)


Initialize the model key
Get the following keys...
model_dir: 
./training_resu//1_model/
model_type: 
6
model_args: 
{'vocab_size': 256, 'n_layer': 36, 'block_size': 1024, 'n_embd': 1024, 'n_head': 16, 'bias': False, 'dropout': 0.2}
Creating the given path...
Done.
Save the model config ...


In [24]:
print (
    f"model dir: "+
    ModelKeys['model_dir']
)

model dir: ./training_resu//1_model/


In [25]:
# ====================================================
# needed for the second run
# ====================================================
# load in the model if this is not first train
#
if CKeys['Working_Mode']>0 and CKeys['IF_FirstRun']>1:
    #
    if CKeys['Resume_from_where'] == 'LAST':
        ckpt_name = TrainKeys['out_dir_last']+'Last_ckpt.pt'
    elif CKeys['Resume_from_where'] == 'BEST':
        ckpt_name = TrainKeys['out_dir_best']+'Best_ckpt.pt'
    #
    print ("CK_PT: ", ckpt_name)
    #
    # prepare to resume training from a checkpoint
    checkpoint = torch.load(ckpt_name, map_location=TrainKeys['device'])
    checkpoint_model_args = checkpoint['model_args']
    #
    # force these config attributes to be equal otherwise we can't even resume training
    # the rest of the attributes (e.g. dropout) can stay as desired from command line
    #
    # For new models, this is just a check
    # |||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
    # problem-specified
    for k in checkpoint_model_args.keys():
        if model_args[k] != checkpoint_model_args[k]:
            print ("hard update needed on ", k)
            print ("old: ", model_args[k])
            print ("new: ", checkpoint_model_args[k])
            model_args[k] = checkpoint_model_args[k]
    print ("Updated model_args: ", model_args)



In [26]:
#
if CKeys['Working_Mode']>0:
    print ("=========================================")
    print ("Initialize the model from the scratch...")
    print ("=========================================")
    #
    model_config = ModelPack.build_model_config(
        CKeys,
        model_args
    )
    #
    model = ModelPack.build_model(
        CKeys,
        model_config,
    )

    # # may add another IF to distinguish the model types
    # #
    # # process: model configuration
    # gptconf = GPTConfig(**model_args)
    # print ("Recieve model config: \n", gptconf)
    # # process: model
    # model = GPT(gptconf)

    print ("\n\n")
    print (model)

Initialize the model from the scratch...
number of parameters: 453.32M



GPT(
  (transformer): ModuleDict(
    (wte): Embedding(256, 1024)
    (wpe): Embedding(1024, 1024)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-35): 36 x MHA_Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=1024, out_features=3072, bias=False)
          (c_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=1024, out_features=4096, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=1024, out_features

In [27]:
print ("===================================================")
print ("Load back the model if this's not the 1st training.")
print ("===================================================")
#
if CKeys['Working_Mode']>0 and CKeys['IF_FirstRun']>1:
    # Here, for GPT, we use checkpoint to load back the model
    print ("Loading the saved model...")
    #
    state_dict = checkpoint['model']
    # NOTE, this one has unwanted contents.
    # fix the keys of the state dictionary :(
    # honestly no idea how checkpoints sometimes get this prefix, have to debug more
    unwanted_prefix = '_orig_mod.'
    for k,v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
    #
    # load back the previous breaking point
    print ("Load in the model...")
    model.load_state_dict(state_dict)
    print ("Update some other recrods...")
    # clean
    state_dict = None
    print ("Done")
    # TBA

Load back the model if this's not the 1st training.


In [28]:

# consider loading back the trained model
print ('===========================================================')
print ('Consider loading back the trained model IF available...')
print ('===========================================================')
#
if CKeys['Working_Mode']>0 and CKeys['IF_FirstRun']>1:
    # Here, for GPT, we use checkpoint to load back the model
    print ("Loading the training history...")
    # ref-code:
    # checkpoint = {
    #                 'model': raw_model.state_dict(),
    #                 'optimizer': optimizer.state_dict(),
    #                 'model_args': model_args,
    #                 'completed_updating_steps': completed_updating_steps,
    #                 'step_num': this_step,
    #                 'iter_num_at_best_loss': GAS_at_best_val_loss,
    #                 'best_val_loss': best_val_loss
    #             }
    best_val_loss_0 = checkpoint['best_val_loss']
    GAS_at_best_val_loss_0 = checkpoint['iter_num_at_best_loss'] # -100
    # finished_steps_0 = checkpoint['iter_num']*TrainKeys['gradient_accumulation_steps']
    finished_steps_0 = checkpoint['step_num']
    completed_updating_steps_0 = checkpoint['completed_updating_steps']
    #
    print ("On the read checkpoint:")
    print (f"step_num: {finished_steps_0}")
    print (f"GAS_num: {completed_updating_steps_0}")
    print (f"best_val_loss: {best_val_loss_0}")
    print (f"GAS_at_best_val_loss: {GAS_at_best_val_loss_0}")
    # TBA
    # # cleaning
    # checkpoint=None

Consider loading back the trained model IF available...


## 5. On training

In [29]:
# load in
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.distributed import init_process_group, destroy_process_group

In [30]:
# Training mode
if CKeys['Working_Mode']==1 and CKeys['IF_FirstRun']==1:

    print ('===========================================================')
    print ('Initialize training at 1st run ...')
    print ('===========================================================')

    TrainKeys = {}
    # pass on the keys
    print ("Get the following keys...")
    for this_key in PKeys['Train_PKeys'].keys():
        print ("{}: \n{}".format(this_key, PKeys['Train_PKeys'][this_key]))
        TrainKeys[this_key] = PKeys['Train_PKeys'][this_key]

    # 1. create dir
    UtilityPack.create_path(TrainKeys['out_dir'])
    # two subdir just for model checkpoint
    TrainKeys['out_dir_last'] = TrainKeys['out_dir'] + "last_check/"
    TrainKeys['out_dir_best'] = TrainKeys['out_dir'] + "best_check/"
    UtilityPack.create_path(TrainKeys['out_dir_last'])
    UtilityPack.create_path(TrainKeys['out_dir_best'])
    #
    # Some that can only be defined here
    TrainKeys['max_iters'] = math.ceil(
        len(train_dataloader)/TrainKeys['gradient_accumulation_steps']
    ) * TrainKeys['num_train_epochs']  # in GAS
    TrainKeys['lr_decay_iters'] = TrainKeys['max_iters']

    # 2. process: problem-specific ones
    # ||||||||||||||||||||||||||||||||||||||||||||||||||||||
    # initialize the recrods
    best_val_loss_0 = 1.E09
    GAS_at_best_val_loss_0 = -100
    finished_steps_0 = 0
    completed_updating_steps_0 = 0

    # TrainKeys['best_val_loss'] = 1.e09
    # TrainKeys['GAS_at_best_val_loss'] = -100

    # =========================================================
    # below are not from PKeys
    # secondary ones
    # files: use fixed ones
    TrainKeys['1_train_loss.log']=TrainKeys['out_dir']+'1_train_loss.log'
    TrainKeys['2_vali_loss.log']=TrainKeys['out_dir']+'2_vali_loss.log'
    TrainKeys['2_vali_gene.log']=TrainKeys['out_dir']+'2_vali_gene.log'
    # # --
    # TrainKeys['3_save_model.log']=TrainKeys['out_dir']+'3_save_model.log'
    # ++
    TrainKeys['3_save_model_last.log']=TrainKeys['out_dir_last']+'3_save_model_last.log'
    TrainKeys['3_save_model_best.log']=TrainKeys['out_dir_best']+'3_save_model_best.log'

    # 3. save the TrainKeys
    print("==========================================")
    print("Save the train config ...")
    print("==========================================")

    # on ModelKeys
    with open(PKeys['pk_train_pack'], 'wb') as handle:
        pickle.dump(TrainKeys, handle, protocol=pickle.HIGHEST_PROTOCOL)

elif CKeys['Working_Mode']==1 and CKeys['IF_FirstRun']>1:

    print("==========================================")
    print ('This is not the first run')
    print ('Load back in the train packages...')
    print ('Done in the previous block with ModelKeys')
    print("==========================================")

else:
    # this a test mode: do whatever needed
    pass

Initialize training at 1st run ...
Get the following keys...
out_dir: 
./training_resu//1_model//training_dir/
batch_size: 
12
block_size: 
208
vocab_size: 
256
num_train_epochs: 
5
gradient_accumulation_steps: 
8
weight_decay: 
0.1
beta1: 
0.9
beta2: 
0.95
grad_clip: 
1.0
warmup_iters: 
1000
learning_rate: 
0.0001
decay_lr: 
True
min_lr: 
1e-05
report_1_trai_loss_this_GAS: 
20
report_2_vali_pred_this_GAS: 
200
report_3_save_mode_this_GAS: 
400
backend: 
nccl
device: 
cuda
dtype: 
bfloat16
compile: 
True
Creating the given path...
Done.
Creating the given path...
Done.
Creating the given path...
Done.
Save the train config ...


In [31]:
if CKeys['Working_Mode']==1:
    print ("Check the status before the preparation...")
    print ("allow_ft32 in cuda: \n",
           torch.backends.cuda.matmul.allow_tf32)
    print ("allow_ft32 in cudnn: \n",
           torch.backends.cudnn.allow_tf32)

Check the status before the preparation...
allow_ft32 in cuda: 
 False
allow_ft32 in cudnn: 
 True


In [32]:

if CKeys['Working_Mode']==1 and CKeys['Debug_Train']==1:
    print ({**TrainKeys, **ModelKeys})
    print (TrainKeys['dtype'])
    print (TrainKeys['compile'])
    # print (TrainKeys['ddp']) # this one comes only after processing

In [33]:
if CKeys['Working_Mode']==1:
    # prepare for training
    # ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
    # This block is problem-specified
    #
    # before ddp, make a record of model+trainer:
    config = {**TrainKeys, **ModelKeys}
    print ("old config: \n", config)
    #
    # XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
    # ATTENSION: afte this step, ranks matter
    # XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
    #
    TrainKeys, ctx = TrainPack.initialize_train_fun(TrainKeys)
    # this one updates the TrainKeys based on whether this is the master_process
    # and create folder if it is master_process
    #
    #
    print ("Is this a DDP run: ", TrainKeys['ddp'])
    #
    # crop down the model block size if desired, using model surgery
    if TrainKeys['block_size'] < model.config.block_size:
        print ("crop down the model block size")
        model.crop_block_size(TrainKeys['block_size'])
        model_args['block_size'] = TrainKeys['block_size'] # so that the checkpoint will have the right value
    #
    # move the model to the device
    model.to(TrainKeys['device'])
    #
    # initialize a GradScaler. If enabled=False scaler is a no-op
    scaler = torch.cuda.amp.GradScaler(enabled=(TrainKeys['dtype'] == 'float16'))
    #
    # optimizer: initilize
    optimizer = model.configure_optimizers(
        TrainKeys['weight_decay'],
        TrainKeys['learning_rate'],
        (TrainKeys['beta1'], TrainKeys['beta2']),
        TrainKeys['device_type']
    )
    # load it back
    if CKeys['IF_FirstRun'] !=1:
        print ("load in optimizer from a previous run")
        optimizer.load_state_dict(checkpoint['optimizer'])
    #
    # initilize for 1st run
    # free up memory if not 1st run;
    checkpoint = None # free up memory
    #
    # compile the model
    if TrainKeys['compile']:
        print("compiling the model... (takes a ~minute)")
        # https://stackoverflow.com/questions/62691279/how-to-disable-tokenizers-parallelism-true-false-warning
        # import os
        os.environ["TOKENIZERS_PARALLELISM"] = "true" # "false"

        unoptimized_model = model
        model = torch.compile(model) # requires PyTorch 2.0
    # wrap model into DDP container
    if TrainKeys['ddp']:
        print ("adjust model through DDP...")
        model = DDP(model, device_ids=[TrainKeys['ddp_local_rank']])
    #
    # loagging
    config = {**TrainKeys, **ModelKeys}
    print ("new config: \n", config)
    # if TrainKeys['wandb_log'] and TrainKeys['master_process']:
    #     import wandb
    #     wandb.init(
    #         project=TrainKeys['wandb_project'],
    #         name=TrainKeys['wandb_run_name'],
    #         config=config,
    #     )

old config: 
 {'out_dir': './training_resu//1_model//training_dir/', 'batch_size': 12, 'block_size': 208, 'vocab_size': 256, 'num_train_epochs': 5, 'gradient_accumulation_steps': 8, 'weight_decay': 0.1, 'beta1': 0.9, 'beta2': 0.95, 'grad_clip': 1.0, 'warmup_iters': 1000, 'learning_rate': 0.0001, 'decay_lr': True, 'min_lr': 1e-05, 'report_1_trai_loss_this_GAS': 20, 'report_2_vali_pred_this_GAS': 200, 'report_3_save_mode_this_GAS': 400, 'backend': 'nccl', 'device': 'cuda', 'dtype': 'bfloat16', 'compile': True, 'out_dir_last': './training_resu//1_model//training_dir/last_check/', 'out_dir_best': './training_resu//1_model//training_dir/best_check/', 'max_iters': 23440, 'lr_decay_iters': 23440, '1_train_loss.log': './training_resu//1_model//training_dir/1_train_loss.log', '2_vali_loss.log': './training_resu//1_model//training_dir/2_vali_loss.log', '2_vali_gene.log': './training_resu//1_model//training_dir/2_vali_gene.log', '3_save_model_last.log': './training_resu//1_model//training_dir/las

/var/tmp/ipykernel_976236/513670383.py:31: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(TrainKeys['dtype'] == 'float16'))


num decayed parameter tensors: 146, with 453,459,968 parameters
num non-decayed parameter tensors: 73, with 74,752 parameters
using fused AdamW: True
compiling the model... (takes a ~minute)
new config: 
 {'out_dir': './training_resu//1_model//training_dir/', 'batch_size': 12, 'block_size': 208, 'vocab_size': 256, 'num_train_epochs': 5, 'gradient_accumulation_steps': 8, 'weight_decay': 0.1, 'beta1': 0.9, 'beta2': 0.95, 'grad_clip': 1.0, 'warmup_iters': 1000, 'learning_rate': 0.0001, 'decay_lr': True, 'min_lr': 1e-05, 'report_1_trai_loss_this_GAS': 20, 'report_2_vali_pred_this_GAS': 200, 'report_3_save_mode_this_GAS': 400, 'backend': 'nccl', 'device': 'cuda', 'dtype': 'bfloat16', 'compile': True, 'out_dir_last': './training_resu//1_model//training_dir/last_check/', 'out_dir_best': './training_resu//1_model//training_dir/best_check/', 'max_iters': 23440, 'lr_decay_iters': 23440, '1_train_loss.log': './training_resu//1_model//training_dir/1_train_loss.log', '2_vali_loss.log': './training_

In [34]:
if CKeys['Working_Mode']==1:
    print ("Check the status After the preparation...")
    print ("allow_ft32 in cuda: \n",
           torch.backends.cuda.matmul.allow_tf32)
    print ("allow_ft32 in cudnn: \n",
           torch.backends.cudnn.allow_tf32)

Check the status After the preparation...
allow_ft32 in cuda: 
 True
allow_ft32 in cudnn: 
 True


In [35]:

if CKeys['Working_Mode']==1:
    print ("IF this is master_process: ", TrainKeys['master_process'])

IF this is master_process:  True


In [36]:
#
# prepare to record the training history
#
if CKeys['Working_Mode']==1 and CKeys['IF_FirstRun'] == 1:
    print ("==============================================")
    print ("perpare for the 1st training run ...")
    print ("==============================================")
    #
    # ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
    # This is problem-specified
    #
    # for 1st run, initialize the logs
    if TrainKeys['master_process']:
        # 1. train loss
        top_line = f"epoch,step,GAS,wei_loss_trai,plain_loss_trai,lr\n"
        UtilityPack.add_one_line_to_file(
            file_name=TrainKeys['1_train_loss.log'],
            this_line=top_line,
            mode='w', # if exitst, erase it
        )
        # 2. eval loss
        # f"epoch: %d, step: %d, GAS: %d, wei_loss/trai: %f, plain_loss/trai: %f, loss/eval: %f, lr: %f\n"
        top_line = f"epoch,step,GAS,wei_loss_trai,plain_loss_trai,loss_eval,lr\n"
        UtilityPack.add_one_line_to_file(
            file_name=TrainKeys['2_vali_loss.log'],
            this_line=top_line,
            mode='w', # if exitst, erase it
        )
        #
        # 3. predict lines
        UtilityPack.add_one_line_to_file(
            file_name=TrainKeys['2_vali_gene.log'],
            this_line='\n',
            mode='w', # if exitst, erase it
        )



perpare for the 1st training run ...


In [37]:

if CKeys['Working_Mode']==1:
    print (device)
    # print (TrainKeys['ddp'])
    # print (ddp)
    print (TrainKeys['gradient_accumulation_steps'])
    print (TrainKeys['device_type'])
    print (TrainKeys['device'])
    print (TrainKeys['warmup_iters'])
    print (TrainKeys['learning_rate'])
    print (TrainKeys['decay_lr'])
    print (TrainKeys['master_process'])

cuda:0
8
cuda
cuda
1000
0.0001
True
True


In [38]:
'''
FORWARD: {Task:Pred001}={Composition:[(Ti):+1.252e+00,(Cr):+1.960e+01,(Mo):+1.801e+00,(Al):+5.823e-01,(Nb):+3.276e+00,(Co):+2.370e-01,(Ni):+5.011e+01]}=>{Property:[MeanYs:+1.089e+03,StdYS:+1.922e+01,DeltaYS:+5.562e+01]}
INVERSE: {Task:Gene001}={Property:[MeanYs:+1.089e+03,StdYS:+1.922e+01,DeltaYS:+5.562e+01]}=>{Composition:[(Ti):+1.252e+00,(Cr):+1.960e+01,(Mo):+1.801e+00,(Al):+5.823e-01,(Nb):+3.276e+00,(Co):+2.370e-01,(Ni):+5.011e+01]}
'''

'\nFORWARD: {Task:Pred001}={Composition:[(Ti):+1.252e+00,(Cr):+1.960e+01,(Mo):+1.801e+00,(Al):+5.823e-01,(Nb):+3.276e+00,(Co):+2.370e-01,(Ni):+5.011e+01]}=>{Property:[MeanYs:+1.089e+03,StdYS:+1.922e+01,DeltaYS:+5.562e+01]}\nINVERSE: {Task:Gene001}={Property:[MeanYs:+1.089e+03,StdYS:+1.922e+01,DeltaYS:+5.562e+01]}=>{Composition:[(Ti):+1.252e+00,(Cr):+1.960e+01,(Mo):+1.801e+00,(Al):+5.823e-01,(Nb):+3.276e+00,(Co):+2.370e-01,(Ni):+5.011e+01]}\n'

In [39]:


if CKeys['Working_Mode']==1:
    # input
    # test prompts for training
    test_prompts_during_train = [
        # Prompt for Prediction Task
        "{Task:Pred001}={Composition:[(Ti):+1.252e+00,(Cr):+1.960e+01,(Mo):+1.801e+00,(Al):+5.823e-01,(Nb):+3.276e+00,(Co):+2.370e-01,(Ni):+5.021e+01]}=>{Property:",
        # Prompt for Inverse Design (Generation) Task
        "{Task:Gene001}={Property:[MeanYs:+1.099e+03,StdYS:+1.922e+01,DeltaYS:+5.562e+01]}=>{Composition:"
    ]
    # update for remaked one
    test_prompts_during_train = [
        # Prompt for Prediction Task
        "{Task:Pred001}={Composition:[(Ti):+1.252e+00,(Cr):+1.960e+01,(Mo):+1.801e+00,(Al):+5.823e-01,(Nb):+3.276e+00,(Co):+2.370e-01,(Ni):+5.021e+01]}=>{Property:",
        # Prompt for Inverse Design (Generation) Task
        "{Task:Gene001}={Property:[MeanYs:+1.099e+03,StdYS:+1.922e+01,DeltaYS:+5.562e+01]}=>{Composition:"
    ]
    # key tokens
    target_keywords = [
        "0","1","2","3","4","5","6","7","8","9",
        ".","+","-","e",
    ]


In [40]:
# prepare training
#
if CKeys['Working_Mode']==1:
    #
    # 1. key tokens
    keytoken_ids = TrainPack.translate_words_w_tokenizers(
        tokenizer,
        target_keywords,
    )
    print (keytoken_ids)
    #
    # 2. translate that into weight
    wei_list_for_all_vocab = TrainPack.build_weight_list_for_vocab(
        model.config.vocab_size,
        keytoken_ids,
        wei_0=1.,
        wei_1=2.,
    ).to(device)
    print (wei_list_for_all_vocab.shape)

[129, 120, 121, 122, 123, 124, 125, 126, 127, 128, 130, 131, 132, 158]
torch.Size([256])


In [41]:
import AlloyGPT.DataPack_v3 as DataPack
from tokenizers import Tokenizer
from tokenizers import decoders

def check_tokenizer_fidelity(tokenizer, original_sentence):
    """
    Encodes a sentence and then decodes it to check if the
    output matches the input.

    Args:
        tokenizer: The initialized tokenizer instance.
        original_sentence (str): The sentence to test.
    """
    print("-" * 50)
    print(f"Original Sentence:\n'{original_sentence}'")
    print("-" * 50)

    # 1. Encode the sentence into token IDs
    encoded_output = tokenizer.encode(original_sentence)
    token_ids = encoded_output.ids
    
    print(f"Encoded Token IDs:\n{token_ids}\n")
    print(f"Number of tokens: {len(token_ids)}\n")

    # 2. Decode the token IDs back into a sentence
    decoded_sentence = tokenizer.decode(token_ids)

    print(f"Decoded Sentence:\n'{decoded_sentence}'")
    print("-" * 50)

    # 3. Compare and report the result
    if original_sentence == decoded_sentence:
        print("✅ SUCCESS: The decoded sentence perfectly matches the original sentence.")
    else:
        print("❌ FAILURE: The decoded sentence does NOT match the original.")
        print(f"Length of original: {len(original_sentence)}")
        print(f"Length of decoded:  {len(decoded_sentence)}")

    print("=" * 50 + "\n")


if __name__ == '__main__':
    # --- Configuration ---
    # These parameters should match what you use in your main notebook.
    tokenizer_type = 'BPE_customerized_0'
    # IMPORTANT: Make sure this path is correct relative to where you run the script.
    # You might need to adjust it to '../assets/...' or a full path if needed.
    tokenizer_file = './assets/cBPE_tokenizer_ver1.json' 
    seq_len = 208 # The sequence length used for padding.

    # --- Load Tokenizer ---
    print(f"Loading tokenizer from: {tokenizer_file}\n")
    try:
        tokenizer = DataPack.build_tokenizer(tokenizer_type, tokenizer_file, seq_len)
    except Exception as e:
        print(f"Error loading tokenizer: {e}")
        print("Please ensure the file path is correct and the AlloyGPT package is in your Python path.")
        exit()

    # --- Define Sample Sentence ---
    # A "good" sample from your validation logs to test the round trip.
    sample_sentence_1 = "{Task:Pred001}={Composition:[(Ti):+1.252e+00,(Cr):+1.960e+01,(Mo):+1.801e+00,(Al):+5.823e-01,(Nb):+3.276e+00,(Co):+2.370e-01,(Ni):+5.021e+01]}=>{Property:[MeanYs:+1.244e+03,StdYS:+2.271e+01,DeltaYS:+6.297e+01]}</s>"
    
    # Another sample for the inverse task
    sample_sentence_2 = "{Task:Gene001}={Property:[MeanYs:+1.099e+03,StdYS:+1.922e+01,DeltaYS:+5.562e+01]}=>{Composition:[(Ti):+7.998e-01,(Cr):+1.931e+01,(Mo):+1.720e+00,(Al):+5.553e-01,(Nb):+3.178e+00,(Co):+7.409e-01,(Ni):+5.159e+01]}</s>"

    # --- Run the Checks ---
    check_tokenizer_fidelity(tokenizer, sample_sentence_1)
    check_tokenizer_fidelity(tokenizer, sample_sentence_2)


Loading tokenizer from: ./assets/cBPE_tokenizer_ver1.json

--------------------------------------------------
Original Sentence:
'{Task:Pred001}={Composition:[(Ti):+1.252e+00,(Cr):+1.960e+01,(Mo):+1.801e+00,(Al):+5.823e-01,(Nb):+3.276e+00,(Co):+2.370e-01,(Ni):+5.021e+01]}=>{Property:[MeanYs:+1.244e+03,StdYS:+2.271e+01,DeltaYS:+6.297e+01]}</s>'
--------------------------------------------------
Encoded Token IDs:
[133, 199, 154, 172, 164, 138, 195, 171, 158, 157, 129, 129, 120, 134, 139, 133, 182, 168, 166, 169, 168, 172, 162, 173, 162, 168, 167, 138, 135, 23, 138, 131, 120, 130, 121, 124, 121, 158, 131, 129, 129, 137, 25, 138, 131, 120, 130, 128, 125, 129, 158, 131, 129, 120, 137, 43, 138, 131, 120, 130, 127, 129, 120, 158, 131, 129, 129, 137, 14, 138, 131, 124, 130, 127, 121, 122, 158, 132, 129, 120, 137, 42, 138, 131, 122, 130, 121, 126, 125, 158, 131, 129, 129, 137, 28, 138, 131, 121, 130, 122, 126, 129, 158, 132, 129, 120, 137, 29, 138, 131, 124, 130, 129, 121, 120, 158, 131, 129, 

In [42]:
# Run this code again now that your sentences include the </s> token

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def get_true_token_length(example):
    pred_encoding = tokenizer.encode(example['Pred001_sentence']) # Now includes </s>
    gene_encoding = tokenizer.encode(example['Gene001_sentence']) # Now includes </s>
    return {
        'pred_len': sum(pred_encoding.attention_mask),
        'gene_len': sum(gene_encoding.attention_mask)
    }

dataset_with_lengths = sentence_dataset_dict_sepe['train'].map(get_true_token_length)
max_len = max(max(dataset_with_lengths['pred_len']), max(dataset_with_lengths['gene_len']))
new_block_size = (max_len + 15) & -16 

print(f"The longest sentence (including EOS) has {max_len} tokens.")
print(f"RECOMMENDED new block_size: {new_block_size}")

The longest sentence (including EOS) has 189 tokens.
RECOMMENDED new block_size: 192


In [ ]:
if CKeys['Working_Mode'] == 1:
    # for some 
    import torch._dynamo
    torch._dynamo.config.suppress_errors = True
    
    print ("Start the training loop...")
    eval_size = 20

    TrainPack.training_loop_AR_LM(
        #
        model,
        tokenizer,
        ctx,
        optimizer,
        TrainKeys,
        train_dataloader,
        eval_dataloader,
        eval_size,
        wei_list_for_all_vocab,
        scaler,
        test_prompts_during_train,
        #
        best_val_loss_0,
        GAS_at_best_val_loss_0,
        finished_steps_0,
        completed_updating_steps_0,
        model_args,
    )

Start the training loop...
####################

epoch: 0, step: 160, GAS: 20, wei_loss/trai: 2.936606, plain_loss/trai: 3.189740, lr: 0.000002

####################

epoch: 0, step: 320, GAS: 40, wei_loss/trai: 1.358576, plain_loss/trai: 1.353947, lr: 0.000004

####################

epoch: 0, step: 480, GAS: 60, wei_loss/trai: 0.702409, plain_loss/trai: 0.570343, lr: 0.000006

####################

epoch: 0, step: 640, GAS: 80, wei_loss/trai: 0.625447, plain_loss/trai: 0.490087, lr: 0.000008

####################

epoch: 0, step: 800, GAS: 100, wei_loss/trai: 0.615976, plain_loss/trai: 0.482708, lr: 0.000010

####################

epoch: 0, step: 960, GAS: 120, wei_loss/trai: 0.591350, plain_loss/trai: 0.462270, lr: 0.000012

####################

epoch: 0, step: 1120, GAS: 140, wei_loss/trai: 0.573733, plain_loss/trai: 0.446596, lr: 0.000014

####################

epoch: 0, step: 1280, GAS: 160, wei_loss/trai: 0.598344, plain_loss/trai: 0.475696, lr: 0.000016

####################

e

 48%|████▊     | 20/42 [00:16<00:18,  1.22it/s]


epoch: 0, step: 1600, GAS: 200, wei_loss/trai: 0.578932, plain_loss/trai: 0.458783, loss/eval: 0.415307, lr: 0.000020

0: 
{Task:Pred001}={Composition:[(Ti):+1.252e+00,(Cr):+1.960e+01,(Mo):+1.801e+00,(Al):+5.823e-01,(Nb):+3.276e+00,(Co):+2.370e-01,(Ni):+5.021e+01]}=>{Property:[MeanYs:+1.843e+03,StdYS:+4.480e+01,De-01,(Ni):+1.m04e+02]}

1: 
{Task:Gene001}={Property:[MeanYs:+1.099e+03,StdYS:+1.922e+01,DeltaYS:+5.562e+01]}=>{Composition:[(Ti):+6.904e+01,(Cr):+1.792e+01,(Mo):+3.259e+00,(Al):+3.59we-01,(Nb):+4.156e+4.309e+01,De-01,(Ni):+1.475e+01]}

####################

epoch: 0, step: 1760, GAS: 220, wei_loss/trai: 0.666561, plain_loss/trai: 0.529255, lr: 0.000022

####################

epoch: 0, step: 1920, GAS: 240, wei_loss/trai: 0.697310, plain_loss/trai: 0.549524, lr: 0.000024

####################

epoch: 0, step: 2080, GAS: 260, wei_loss/trai: 0.498003, plain_loss/trai: 0.378921, lr: 0.000026

####################

epoch: 0, step: 2240, GAS: 280, wei_loss/trai: 0.498655, plain_loss

 48%|████▊     | 20/42 [00:00<00:00, 27.92it/s]


epoch: 0, step: 3200, GAS: 400, wei_loss/trai: 0.472319, plain_loss/trai: 0.363968, loss/eval: 0.365708, lr: 0.000040

0: 
{Task:Pred001}={Composition:[(Ti):+1.252e+00,(Cr):+1.960e+01,(Mo):+1.801e+00,(Al):+5.823e-01,(Nb):+3.276e+00,(Co):+2.370e-01,(Ni):+5.021e+01]}=>{Property:[MeanYs:+1.079e+03,StdYS:+4.417e+01,DeltaYS:+1.066e+03]}

1: 
{Task:Gene001}={Property:[MeanYs:+1.099e+03,StdYS:+1.922e+01,DeltaYS:+5.562e+01]}=>{Composition:[(Ti):+1.118e+00,(Cr):+1.988e+01,(Mo):+2.816e+00,(Al):+6.863e-01,(Nb):+5.885e+00,(Co):+03178e-01,(Ni):+5.661e+01]}


save so-far BEST checkpoint.


save checkpoint REGULARLY at GAS: 400.

####################

epoch: 0, step: 3360, GAS: 420, wei_loss/trai: 0.485598, plain_loss/trai: 0.373926, lr: 0.000042

####################

epoch: 0, step: 3520, GAS: 440, wei_loss/trai: 0.479209, plain_loss/trai: 0.368697, lr: 0.000044

####################

epoch: 0, step: 3680, GAS: 460, wei_loss/trai: 0.479957, plain_loss/trai: 0.368203, lr: 0.000046

#################

...

epoch: 0, step: 6400, GAS: 800, wei_loss/trai: 0.452668, plain_loss/trai: 0.345521, loss/eval: 0.344444, lr: 0.000080

0: 
{Task:Pred001}={Composition:[(Ti):+1.252e+00,(Cr):+1.960e+01,(Mo):+1.801e+00,(Al):+5.823e-01,(Nb):+3.276e+00,(Co):+2.370e-01,(Ni):+5.021e+01]}=>{Property:[MeanYs:+1.394e+03,StdYS:+5.262e+01,DeltaYS:+1.507e+02]}

1: 
{Task:Gene001}={Property:[MeanYs:+1.099e+03,StdYS:+1.922e+01,DeltaYS:+5.562e+01]}=>{Composition:[(Ti):+1.108e+00,(Cr):+1.865e+01,(Mo):+3.149e+00,(Al):+3.089e-01,(Nb):+4.969e+00,(Co):+2.749e-01,(Ni):+5.308e+01]}


save so-far BEST checkpoint.


save checkpoint REGULARLY at GAS: 800.

####################

epoch: 0, step: 6560, GAS: 820, wei_loss/trai: 0.457813, plain_loss/trai: 0.349676, lr: 0.000082

####################

epoch: 0, step: 6720, GAS: 840, wei_loss/trai: 0.453115, plain_loss/trai: 0.346603, lr: 0.000084

####################

epoch: 0, step: 6880, GAS: 860, wei_loss/trai: 0.460687, plain_loss/trai: 0.352318, lr: 0.000086

#################

 48%|████▊     | 20/42 [00:00<00:00, 27.93it/s]


epoch: 0, step: 8000, GAS: 1000, wei_loss/trai: 0.446774, plain_loss/trai: 0.341470, loss/eval: 0.342081, lr: 0.000100

0: 
{Task:Pred001}={Composition:[(Ti):+1.252e+00,(Cr):+1.960e+01,(Mo):+1.801e+00,(Al):+5.823e-01,(Nb):+3.276e+00,(Co):+2.370e-01,(Ni):+5.021e+01]}=>{Property:[MeanYs:+1.216e+03,StdYS:+5.365e+01,DeltaYS:+1.573e+02]}

1: 
{Task:Gene001}={Property:[MeanYs:+1.099e+03,StdYS:+1.922e+01,DeltaYS:+5.562e+01]}=>{Composition:[(Ti):+8.364e-01,(Cr):+1.812e+01,(Mo):+3.037e+00,(Al):+2.971e-01,(Nb):+5.256e+00,(Co):+0.595e-01,(Ni):+5.348e+01]}

####################

epoch: 0, step: 8160, GAS: 1020, wei_loss/trai: 0.442960, plain_loss/trai: 0.338766, lr: 0.000100

####################

epoch: 0, step: 8320, GAS: 1040, wei_loss/trai: 0.446014, plain_loss/trai: 0.341192, lr: 0.000100

####################

epoch: 0, step: 8480, GAS: 1060, wei_loss/trai: 0.449066, plain_loss/trai: 0.343921, lr: 0.000100

####################

epoch: 0, step: 8640, GAS: 1080, wei_loss/trai: 0.440443, plain

 48%|████▊     | 20/42 [00:00<00:00, 27.95it/s]


epoch: 0, step: 9600, GAS: 1200, wei_loss/trai: 0.448156, plain_loss/trai: 0.343843, loss/eval: 0.337253, lr: 0.000100

0: 
{Task:Pred001}={Composition:[(Ti):+1.252e+00,(Cr):+1.960e+01,(Mo):+1.801e+00,(Al):+5.823e-01,(Nb):+3.276e+00,(Co):+2.370e-01,(Ni):+5.021e+01]}=>{Property:[MeanYs:+1.279e+03,StdYS:+4.889e+01,DeltaYS:+1.379e+02]}

1: 
{Task:Gene001}={Property:[MeanYs:+1.099e+03,StdYS:+1.922e+01,DeltaYS:+5.562e+01]}=>{Composition:[(Ti):+9.143e-01,(Cr):+1.930e+01,(Mo):+3.359e+00,(Al):+3.287e-01,(Nb):+5.479e+00,(Co):+3.870e-02,(Ni):+5.209e+01]}

